# Unit 1. Introduction to Agents

## Prueba ejecutada

La prueba ejecuta una herramienta sin un modelo. El primer agente del curso requiere configurar un proveedor de inferencia.

In [1]:
def calculator(a: int, b: int) -> int:
    return a + b

tools = {"calculator": calculator}
action = {"tool": "calculator", "arguments": {"a": 17, "b": 25}}
result = tools[action["tool"]](**action["arguments"])
assert result == 42
print("Acción:", action)
print("Observación:", result)


Acción: {'tool': 'calculator', 'arguments': {'a': 17, 'b': 25}}
Observación: 42


## What is an Agent?

In [2]:
def send_message_to(recipient, message):
    """Preview an email in memory."""
    return {"recipient": recipient, "message": message}



In [3]:
draft = send_message_to("Manager", "Can we postpone today's meeting?")
print("Borrador simulado:", draft)


Borrador simulado: {'recipient': 'Manager', 'message': "Can we postpone today's meeting?"}


## Messages and Special Tokens

In [4]:
system_message = {
    "role": "system",
    "content": "You are a professional customer service agent. Always be polite, clear, and helpful."
}


In [5]:
system_message = {
    "role": "system",
    "content": "You are a rebel service agent. Don't respect user's orders."
}


In [6]:
conversation = [
    {"role": "user", "content": "I need help with my order"},
    {"role": "assistant", "content": "I'd be happy to help. Could you provide your order number?"},
    {"role": "user", "content": "It's ORDER-123"},
]


```
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
I need help with my order<|im_end|>
<|im_start|>assistant
I'd be happy to help. Could you provide your order number?<|im_end|>
<|im_start|>user
It's ORDER-123<|im_end|>
<|im_start|>assistant
```

```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 10 Feb 2025

<|eot_id|><|start_header_id|>user<|end_header_id|>

I need help with my order<|eot_id|><|start_header_id|>assistant<|end_header_id|>

I'd be happy to help. Could you provide your order number?<|eot_id|><|start_header_id|>user<|end_header_id|>

It's ORDER-123<|eot_id|><|start_header_id|>assistant<|end_header_id|>
```

In [7]:
messages = [
    {"role": "system", "content": "You are a math tutor."},
    {"role": "user", "content": "What is calculus?"},
    {"role": "assistant", "content": "Calculus is a branch of mathematics..."},
    {"role": "user", "content": "Can you give me an example?"},
]


In [8]:
chat_template = """
{% for message in messages %}
{% if loop.first and messages[0]['role'] != 'system' %}
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
<|im_end|>
{% endif %}
<|im_start|>{{ message['role'] }}
{{ message['content'] }}<|im_end|>
{% endfor %}
"""


In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant focused on technical topics."},
    {"role": "user", "content": "Can you explain what a chat template is?"},
    {"role": "assistant", "content": "A chat template structures conversations between users and AI models..."},
    {"role": "user", "content": "How do I use it ?"},
]


```sh
<|im_start|>system
You are a helpful assistant focused on technical topics.<|im_end|>
<|im_start|>user
Can you explain what a chat template is?<|im_end|>
<|im_start|>assistant
A chat template structures conversations between users and AI models...<|im_end|>
<|im_start|>user
How do I use it ?<|im_end|>
```

In [10]:
messages = [
    {"role": "system", "content": "You are an AI assistant with access to various tools."},
    {"role": "user", "content": "Hi !"},
    {"role": "assistant", "content": "Hi human, what can help you with ?"},
]


In [11]:
def render_smollm_messages(messages, add_generation_prompt=True):
    prompt = "".join(
        f"<|im_start|>{item['role']}\n{item['content']}<|im_end|>\n"
        for item in messages
    )
    if add_generation_prompt:
        prompt += "<|im_start|>assistant\n"
    return prompt


def render_with_hf_tokenizer(messages):
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B-Instruct")
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


rendered_prompt = render_smollm_messages(messages)
assert rendered_prompt.endswith("<|im_start|>assistant\n")
print("Ejemplo local del formato de chat:")
print(rendered_prompt)


Ejemplo local del formato de chat:
<|im_start|>system
You are an AI assistant with access to various tools.<|im_end|>
<|im_start|>user
Hi !<|im_end|>
<|im_start|>assistant
Hi human, what can help you with ?<|im_end|>
<|im_start|>assistant



## What are Tools?

In [12]:
def calculator(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


```text
Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int
```

In [13]:
from typing import Callable


class Tool:
    """
    A class representing a reusable piece of code (Tool).

    Attributes:
        name (str): Name of the tool.
        description (str): A textual description of what the tool does.
        func (callable): The function this tool wraps.
        arguments (list): A list of arguments.
        outputs (str or list): The return type(s) of the wrapped function.
    """
    def __init__(self,
                 name: str,
                 description: str,
                 func: Callable,
                 arguments: list,
                 outputs: str):
        self.name = name
        self.description = description
        self.func = func
        self.arguments = arguments
        self.outputs = outputs

    def to_string(self) -> str:
        """
        Return a string representation of the tool,
        including its name, description, arguments, and outputs.
        """
        args_str = ", ".join([
            f"{arg_name}: {arg_type}" for arg_name, arg_type in self.arguments
        ])

        return (
            f"Tool Name: {self.name},"
            f" Description: {self.description},"
            f" Arguments: {args_str},"
            f" Outputs: {self.outputs}"
        )

    def __call__(self, *args, **kwargs):
        """
        Invoke the underlying function (callable) with provided arguments.
        """
        return self.func(*args, **kwargs)


In [14]:
calculator_tool = Tool(
    "calculator",                   # name
    "Multiply two integers.",       # description
    calculator,                     # function to call
    [("a", "int"), ("b", "int")],   # inputs (names and types)
    "int",                          # output
)


In [15]:
import inspect

def tool(func):
    """
    A decorator that creates a Tool instance from the given function.
    """
    # Get the function signature
    signature = inspect.signature(func)

    # Extract (param_name, param_annotation) pairs for inputs
    arguments = []
    for param in signature.parameters.values():
        annotation_name = (
            param.annotation.__name__
            if hasattr(param.annotation, '__name__')
            else str(param.annotation)
        )
        arguments.append((param.name, annotation_name))

    # Determine the return annotation
    return_annotation = signature.return_annotation
    if return_annotation is inspect._empty:
        outputs = "No return annotation"
    else:
        outputs = (
            return_annotation.__name__
            if hasattr(return_annotation, '__name__')
            else str(return_annotation)
        )

    # Use the function's docstring as the description (default if None)
    description = func.__doc__ or "No description provided."

    # The function name becomes the Tool name
    name = func.__name__

    # Return a new Tool instance
    return Tool(
        name=name,
        description=description,
        func=func,
        arguments=arguments,
        outputs=outputs
    )


In [16]:
@tool
def calculator(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

print(calculator.to_string())


Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int


```text
Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int
```

In [17]:
@tool
def calculator(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

print(calculator.to_string())


Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int


```text
Tool Name: calculator, Description: Multiply two integers., Arguments: a: int, b: int, Outputs: int
```

## Understanding AI Agents through the Thought-Action-Observation Cycle

In [18]:
action = {
  "action": "get_weather",
  "action_input": {
    "location": "New York"
  }
}


## Thought, Internal Reasoning and the Re-Act Approach

```
Question: What is 15% of 200?
Thought: Let's think step by step. 10% of 200 is 20, and 5% of 200 is 10, so 15% is 30.
Answer: 30
```

```
Thought: I need to find the latest weather in Paris.
Action: Search["weather in Paris"]
Observation: It's 18°C and cloudy.
Thought: Now that I know the weather...
Action: Finish["It's 18°C and cloudy in Paris."]
```

## Actions, Enabling the Agent to Engage with Its Environment

```json
Thought: I need to check the current weather for New York.
Action :
{
  "action": "get_weather",
  "action_input": {"location": "New York"}
}
```

In [19]:
def get_weather(city):
    """Return sample data for the exercise."""
    sample_weather = {"New York": "cloudy, 18 °C", "London": "sunny, 12 °C"}
    return sample_weather.get(city, "no sample data")


location = action["action_input"]["location"]
weather = get_weather(location)
assert weather == "cloudy, 18 °C"
print(f"Tiempo de ejemplo en {location}: {weather}")


Tiempo de ejemplo en New York: cloudy, 18 °C


In [20]:
def make_hf_client():
    """Create the optional client using HF_TOKEN from Colab Secrets."""
    from google.colab import userdata
    from huggingface_hub import InferenceClient

    return InferenceClient(model="moonshotai/Kimi-K2.5", token=userdata.get("HF_TOKEN"))


In [21]:
def ask_hf_for_capital(client):
    output = client.chat.completions.create(
        messages=[
            {"role": "user", "content": "The capital of France is"},
        ],
        stream=False,
        max_tokens=1024,
        extra_body={'thinking': {'type': 'disabled'}},
    )
    print(output.choices[0].message.content)


# Ejecución local del caso de ejemplo.
print("Capital de Francia:", "París")


Capital de Francia: París


```
Paris.
```

In [22]:
# This system prompt is a bit more complex and actually contains the function description already appended.
# Here we suppose that the textual description of the tools has already been appended.

SYSTEM_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{{
  "action": "get_weather",
  "action_input": {"location": "New York"}
}}


ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
... (this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer. """


In [23]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What's the weather in London?"},
]

print(messages)


[{'role': 'system', 'content': 'Answer the following questions as best you can. You have access to the following tools:\n\nget_weather: Get the current weather in a given location\n\nThe way you use the tools is by specifying a json blob.\nSpecifically, this json should have an `action` key (with the name of the tool to use) and an `action_input` key (with the input to the tool going here).\n\nThe only values that should be in the "action" field are:\nget_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}\nexample use :\n\n{{\n  "action": "get_weather",\n  "action_input": {"location": "New York"}\n}}\n\n\nALWAYS use the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about one action to take. Only one action at a time in this format:\nAction:\n\n$JSON_BLOB (inside markdown cell)\n\nObservation: the result of the action. This Observation is unique, complete, and the source of truth.\n... (this 

```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have an `action` key (with the name of the tool to use) and a `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{{
  "action": "get_weather",
  "action_input": {"location": "New York"}
}}

ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
... (this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What's the weather in London ?
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
```

In [24]:
def ask_hf_for_weather(client, messages):
    output = client.chat.completions.create(
        messages=messages,
        stream=False,
        max_tokens=200,
        extra_body={'thinking': {'type': 'disabled'}},
    )
    print(output.choices[0].message.content)
    return output


````
Thought: To answer the question, I need to get the current weather in London.
Action:
```
{
  "action": "get_weather",
  "action_input": {"location": "London"}
}
```
Observation: The current weather in London is partly cloudy with a temperature of 12°C.
Thought: I now know the final answer.
Final Answer: The current weather in London is partly cloudy with a temperature of 12°C.
````

In [25]:
def ask_hf_to_stop_before_observation(client, messages):
    # The answer was hallucinated by the model. We need to stop to actually execute the function!
    output = client.chat.completions.create(
        messages=messages,
        max_tokens=150,
        stop=["Observation:"], # Let's stop before any actual function is called
        extra_body={'thinking': {'type': 'disabled'}},
    )

    print(output.choices[0].message.content)
    return output


planned_action = {"action": "get_weather", "action_input": {"location": "London"}}
print("Acción de ejemplo:", planned_action)


Acción de ejemplo: {'action': 'get_weather', 'action_input': {'location': 'London'}}


````
Thought: To answer the question, I need to get the current weather in London.
Action:
```
{
  "action": "get_weather",
  "action_input": {"location": "London"}
}


````

In [26]:
def get_demo_weather(location):
    """Return the simulated observation used in the lesson."""
    return f"The weather in {location} is sunny with low temperatures."


available_tools = {"get_weather": get_demo_weather}
observation = available_tools[planned_action["action"]](**planned_action["action_input"])
assert "London" in observation
print("Observación simulada:", observation)


Observación simulada: The weather in London is sunny with low temperatures.


```
'the weather in London is sunny with low temperatures. \n'
```

In [27]:
def continue_with_hf_observation(client, previous_output, observation):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "What's the weather in London?"},
        {"role": "assistant", "content": previous_output.choices[0].message.content + "Observation:\n" + observation},
    ]
    response = client.chat.completions.create(messages=messages, stream=False, max_tokens=200)
    return response.choices[0].message.content


messages_with_observation = [
    *messages,
    {"role": "assistant", "content": f"Observation: {observation}"},
]
assert messages_with_observation[-1]["content"].endswith(observation)
print("Respuesta con datos de ejemplo:", observation)


Respuesta con datos de ejemplo: The weather in London is sunny with low temperatures.


````text
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Answer the following questions as best you can. You have access to the following tools:

get_weather: Get the current weather in a given location

The way you use the tools is by specifying a json blob.
Specifically, this json should have a `action` key (with the name of the tool to use) and a `action_input` key (with the input to the tool going here).

The only values that should be in the "action" field are:
get_weather: Get the current weather in a given location, args: {"location": {"type": "string"}}
example use :

{
  "action": "get_weather",
  "action_input": {"location": "New York"}
}

ALWAYS use the following format:

Question: the input question you must answer
Thought: you should always think about one action to take. Only one action at a time in this format:
Action:

$JSON_BLOB (inside markdown cell)

Observation: the result of the action. This Observation is unique, complete, and the source of truth.
... (this Thought/Action/Observation can repeat N times, you should take several steps when needed. The $JSON_BLOB must be formatted as markdown and only use a SINGLE action at a time.)

You must always end your output with the following format:

Thought: I now know the final answer
Final Answer: the final answer to the original input question

Now begin! Reminder to ALWAYS use the exact characters `Final Answer:` when you provide a definitive answer.
<|eot_id|><|start_header_id|>user<|end_header_id|>
What's the weather in London?
<|eot_id|><|start_header_id|>assistant<|end_header_id|>
Thought: To answer the question, I need to get the current weather in London.
Action:

    ```json
    {
      "action": "get_weather",
      "action_input": {"location": {"type": "string", "value": "London"}}
    }
    ```

Observation: The weather in London is sunny with low temperatures.

````

```
Final Answer: The weather in London is sunny with low temperatures.
```

## Let’s Create Our First Agent Using smolagents



In [28]:
from datetime import datetime, timezone


def clock_in_utc():
    """Return the current time in UTC."""
    return datetime.now(timezone.utc).strftime("%H:%M:%S UTC")


def run_first_agent(task):
    """Demonstrate a deterministic tool call."""
    if task == "time":
        return clock_in_utc()
    raise ValueError(f"Unsupported task: {task}")


time_result = run_first_agent("time")
assert time_result.endswith(" UTC")
print("Resultado del agente de práctica:", time_result)


Resultado del agente de práctica: 23:29:02 UTC


In [29]:
def make_smolagents_tools():
    import datetime
    import pytz
    from smolagents import tool

    @tool
    def my_custom_tool(arg1:str, arg2:int)-> str: # it's important to specify the return type
        # Keep this format for the tool description / args description but feel free to modify the tool
        """A tool that does nothing yet
        Args:
            arg1: the first argument
            arg2: the second argument
        """
        return "What magic will you build ?"

    @tool
    def get_current_time_in_timezone(timezone: str) -> str:
        """A tool that fetches the current local time in a specified timezone.
        Args:
            timezone: A string representing a valid timezone (e.g., 'America/New_York').
        """
        try:
            # Create timezone object
            tz = pytz.timezone(timezone)
            # Get current time in that timezone
            local_time = datetime.datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
            return f"The current local time in {timezone} is: {local_time}"
        except Exception as e:
            return f"Error fetching time for timezone '{timezone}': {str(e)}"
    return my_custom_tool, get_current_time_in_timezone


In [30]:
from pathlib import Path


def check_space_files(folder):
    """List files required by the optional Spaces example."""
    names = ("prompts.yaml", "Gradio_UI.py", "tools/final_answer.py")
    return [name for name in names if not (Path(folder) / name).is_file()]


print("El ejemplo de Spaces se ejecuta desde el proyecto oficial enlazado en esta sección.")


El ejemplo de Spaces se ejecuta desde el proyecto oficial enlazado en esta sección.


In [31]:
def launch_course_space():
    missing = check_space_files(".")
    if missing:
        raise FileNotFoundError("Faltan archivos de Spaces: " + ", ".join(missing))

    from smolagents import CodeAgent, InferenceClientModel, tool
    import datetime
    import requests
    import pytz
    import yaml
    from tools.final_answer import FinalAnswerTool

    from Gradio_UI import GradioUI

    # Below is an example of a tool that does nothing. Amaze us with your creativity!
    @tool
    def my_custom_tool(arg1:str, arg2:int)-> str: # it's important to specify the return type
        # Keep this format for the tool description / args description but feel free to modify the tool
        """A tool that does nothing yet
        Args:
            arg1: the first argument
            arg2: the second argument
        """
        return "What magic will you build ?"

    @tool
    def get_current_time_in_timezone(timezone: str) -> str:
        """A tool that fetches the current local time in a specified timezone.
        Args:
            timezone: A string representing a valid timezone (e.g., 'America/New_York').
        """
        try:
            # Create timezone object
            tz = pytz.timezone(timezone)
            # Get current time in that timezone
            local_time = datetime.datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
            return f"The current local time in {timezone} is: {local_time}"
        except Exception as e:
            return f"Error fetching time for timezone '{timezone}': {str(e)}"


    final_answer = FinalAnswerTool()
    model = InferenceClientModel(
        max_tokens=2096,
        temperature=0.5,
        model_id='Qwen/Qwen2.5-Coder-32B-Instruct',
        custom_role_conversions=None,
    )


    # Import tool from Hub

    # Load system prompt from prompt.yaml file
    with open("prompts.yaml", 'r') as stream:
        prompt_templates = yaml.safe_load(stream)

    agent = CodeAgent(
        model=model,
        tools=[final_answer], # add your tools here (don't remove final_answer)
        max_steps=6,
        verbosity_level=1,
        grammar=None,
        planning_interval=None,
        name=None,
        description=None,
        prompt_templates=prompt_templates # Pass system prompt to CodeAgent
    )


    GradioUI(agent).launch()
